In [1]:
import pandas as pd 
from sqlalchemy import create_engine
import time
import os
import logging
import yfinance as yf

In [2]:
logging.basicConfig(
    filename='../logs/pipeline.log',
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)

logging.info("Fetching gold data...")

In [3]:
engine = create_engine('sqlite:///inventory.db')

In [4]:
# logging.info("Fetching data...")
# gold = yf.download("GC=F", start="2025-10-01")
# gold.to_sql("gold prices", con=engine, if_exists='replace', index=True)
# logging.info("ingestion complete")

In [5]:
def fetch_data(ticker, name):
    """fetch data from yfinance"""
    try:
        logging.info(f"Fetching data from {name}({ticker})")
        
        df = yf.download(ticker, start='2025-10-10')
        
        if df.empty:
            logging.warning(f"No data returned from {name}")
        else:
            logging.info(f"fetched {len(df)} rows of {name}")
        return df
    
    except Exception as e:
        logging.error(f"Error fetching {name}: {e}")
        return None

In [6]:
def store_data(df, table_name):
    """store dataframe into sqlite3"""
    try:
        
        #fix data coloumn
        df = df.reset_index(names='Date')
        
        #fflatten columns
        df.columns = [col[0] if isinstance(col, table) else col for col in df.columns]
        
        df.to_sql(table_name, con=engine, if_exists="replace", index=True)
        logging.info(f"stored data in table: {table_name}")
        
    except Exception as e:
        logging.error(f"Error storing {table_name}: {e}")

In [7]:
def run_pipeline():
    start = time.time()
    
    commodities = {
        "GC=F": "gold",
        "SI=F": "silver",
        "CL=F": "oil"
    }
    
    for ticker, name in commodities.items():
        df = fetch_data(ticker, name)
        
        if df is not None:
            store_data(df, name)
            
    end = time.time()
    total_time = (end-start)/60
    
    logging.info("PIPELINE COMPLETED")
    logging.info(f"Total time taken: {total_time: .2f} minutes")
    
if __name__ == "__main__":
    run_pipeline()

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


In [8]:
from sqlalchemy import text

with engine.connect() as conn:
    conn.execute(text("DROP TABLE IF EXISTS gold"))